***
Monthly Congestion
***

Configuration

In [7]:
export=False

from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import numpy as np
import plotly.express as px
import clean_names
import time
import sys

PATH_GIT = Path.home() / 'Documents' / 'Github Repos' / 'Regional-Monitoring' / 'Indicator_Gen'
PATH_CODE    = PATH_GIT / 'Data' / 'RITIS'
PATH_CONFIG0 = PATH_GIT / 'config'
PATH_CONFIG  = PATH_CODE / 'config'
PATH_SQL = PATH_GIT / 'Data' / 'RITIS' / 'sql_scripts'#not needed because I wont be running the sql scripts in this script

sys.path.append(str(PATH_CONFIG0))


pd.set_option('display.max_columns', None)

# I Drive Path to monthly csv files
PATH_IDRIVE = Path(r"I:/Projects/Josh/Regional Monitoring/Congestion/monthly csv")

# Output Paths
PATH_FINAL = Path(r"I:/Projects/Josh/Regional Monitoring/Congestion/final csv")
PATH_SUMMARY = Path(r"I:/Projects/Josh/Regional Monitoring/Congestion/summary csv")

# SharePoint
PATH_SP = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' 
PATH_CONGESTION = PATH_SP / 'Data' / 'Safe Equitable Resilient Infrastructure' / 'Congestion'
PATH_PHED  = PATH_CONGESTION / 'RITIS' / 'PHED'
PATH_LOTTR = PATH_CONGESTION / 'RITIS' / 'LOTTR'
PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")

# Truck, Pax, or Combined dictionary
tp_dict = {
    'Truck': 'T',
    'Pax': 'P',
    'Combined': 'TAP'
}


***
Clean file names
***

In [8]:
#clean_names.clean_files(PATH_IDRIVE)

***
Load Data
***

In [9]:
# Option 1: Select year, month,and whether it's Truck, Pax, or Combined
year_month = '2025_01'  # Format: 'YYYY_MM'
truck_pax_combined = 'Combined'  # Options: 'Truck', 'Pax', 'Combined'

# Construct file names
zip_name=  year_month + tp_dict[truck_pax_combined]+'.zip'
speed_file = year_month+'.csv'
tmc_file = 'TMC_Identification.csv'

#path for i drive
zip_path = PATH_IDRIVE / zip_name


# Load traffic speed data (equivalent to npmrds_2024_alltmc_paxtruck_comb)
with ZipFile(zip_path) as z:
    with z.open(speed_file) as f:
        df_traffic = pd.read_csv(f)
        df_traffic['measurement_tstamp'] = pd.to_datetime(df_traffic['measurement_tstamp'])

# Load TMC metadata (equivalent to npmrds_2024_alltmc_txt)
with ZipFile(zip_path) as z:
    with z.open(tmc_file) as f:
        df_tmc = pd.read_csv(f)

print(f"Traffic records loaded: {len(df_traffic):,}")
print(f"TMC segments loaded: {len(df_tmc):,}")


Traffic records loaded: 22,873,536
TMC segments loaded: 7,686


In [10]:
df_tmc.head()
df_traffic.head()

,tmc_code,measurement_tstamp,speed,historical_average_speed,reference_speed,travel_time_seconds,data_density,NPMRDS2 2025
0,105P17071,2025-01-01 00:00:00,NaN,NaN,34.0,NaN,NaN,NaN
1,105P17071,2025-01-01 00:15:00,NaN,NaN,34.0,NaN,NaN,NaN
2,105P17071,2025-01-01 00:30:00,NaN,NaN,34.0,NaN,NaN,NaN
3,105P17071,2025-01-01 00:45:00,NaN,NaN,34.0,NaN,NaN,NaN
4,105P17071,2025-01-01 01:00:00,NaN,NaN,34.0,NaN,NaN,NaN


In [11]:
# Free-flow period parameters
FF_PERIOD_START = 20  # Free-flow period starts at or after this hour (8 PM)
FF_PERIOD_END = 6     # Free-flow period ends before this hour (6 AM)

# Weekdays
WEEKDAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

***
Calculating Free Flow Speeds
***

In [12]:
# Join traffic data with TMC metadata
df_ff = df_tmc.merge(df_traffic, left_on='tmc', right_on='tmc_code', how='inner')

# Add hour column
df_ff['hour'] = df_ff['measurement_tstamp'].dt.hour

# Filter for overnight free-flow period (8pm-6am)
ff_mask = (df_ff['hour'] >= FF_PERIOD_START) | (df_ff['hour'] < FF_PERIOD_END)
df_ff = df_ff[ff_mask]

# Filter for NHS roads only
df_ff = df_ff[df_ff['nhs'] > 0]

# Calculate free-flow speed by TMC
# Freeways (f_system 1,2): 85th percentile
# Arterials (other): 60th percentile
def calc_freeflow_speed(group):
    f_system = group['f_system'].iloc[0]
    if pd.notna(f_system) and f_system in [1, 2]:
        return group['speed'].quantile(0.85)
    else:
        return group['speed'].quantile(0.60)

ff_speeds = df_ff.groupby('tmc_code', group_keys=False).apply(calc_freeflow_speed, include_groups=False).reset_index()
ff_speeds.columns = ['tmc_code', 'ff_speed_art60thp']

# Count overnight epochs per TMC (Temp Table #offpk_85th_epochs)
epochs_night = df_ff.groupby('tmc_code').size().reset_index(name='epochs_night')

print(f"Free-flow speeds calculated for {len(ff_speeds):,} TMCs")

Free-flow speeds calculated for 2,905 TMCs


*** 
Calculate Hourly Speeds (Temp Table #avspd_x_tmc_hour)
Note: This data has 15-minute intervals (4 epochs/hour, not 12)
***

In [13]:
# Join traffic data with TMC metadata and filter for weekdays
df_hourly = df_tmc.merge(df_traffic, left_on='tmc', right_on='tmc_code', how='inner')
df_hourly['day_name'] = df_hourly['measurement_tstamp'].dt.day_name()
df_hourly = df_hourly[df_hourly['day_name'].isin(WEEKDAYS)]
df_hourly['hour'] = df_hourly['measurement_tstamp'].dt.hour

# Join with free-flow speeds
df_hourly = df_hourly.merge(ff_speeds, on='tmc_code', how='inner')
# Filter out zero/null speeds BEFORE aggregation
df_hourly_clean = df_hourly[df_hourly['speed'] > 0].copy()
print(f"Valid speed records: {len(df_hourly_clean):,} (removed {len(df_hourly) - len(df_hourly_clean):,} null/zero speeds)")

# Calculate harmonic average speed by TMC and hour
hourly_stats = df_hourly_clean.groupby(['tmc_code', 'hour']).agg(
    total_epochs_hr=('measurement_tstamp', 'count'),
    havg_spd_weekdy=('speed', lambda x: len(x) / (1.0 / x).sum()),
    avg_tt_sec_weekdy=('travel_time_seconds', 'mean'),
    ff_speed_art60thp=('ff_speed_art60thp', 'first')
).reset_index()

# Check what we have
print(f"\nAfter aggregation: {len(hourly_stats):,} TMC-hour combinations")
print(f"Unique TMCs: {hourly_stats['tmc_code'].nunique():,}")
print(f"\nEpoch count statistics:")
print(hourly_stats['total_epochs_hr'].describe())

# Calculating min epochs for each hour across all weekdays in the month by tmc
MIN_EPOCHS = 35  # About 9-10 weekdays worth of data we go with 35

Valid speed records: 4,326,566 (removed 2,087,674 null/zero speeds)

After aggregation: 68,556 TMC-hour combinations
Unique TMCs: 2,900

Epoch count statistics:
count    68556.000000
mean        63.109954
std         31.564817
min          1.000000
25%         35.000000
50%         78.000000
75%         91.000000
max         92.000000
Name: total_epochs_hr, dtype: float64


In [14]:
px.histogram(hourly_stats["total_epochs_hr"]).show()

# Or use adaptive based on your data
MIN_EPOCHS = round(int(hourly_stats['total_epochs_hr'].quantile(0.20)),0)
print(f"\nUsing minimum epoch threshold: {MIN_EPOCHS}")

# Apply filter
hourly_stats = hourly_stats[hourly_stats['total_epochs_hr'] >= MIN_EPOCHS]
print(f"After filtering: {len(hourly_stats):,} rows, {hourly_stats['tmc_code'].nunique():,} TMCs")

# If we still have data, continue with calculations
if len(hourly_stats) > 0:
    # Calculate congestion ratio
    hourly_stats['cong_ratio_hr_weekdy'] = (
        hourly_stats['havg_spd_weekdy'] / hourly_stats['ff_speed_art60thp']
    )
    
    # Rank hours by congestion
    hourly_stats['hour_cong_rank'] = (
        hourly_stats.groupby('tmc_code')['cong_ratio_hr_weekdy']
        .rank(method='first', ascending=True)
    )
    
    print(f"✓ Hourly statistics calculated successfully!")
    print(f"Hourly statistics calculated for {hourly_stats['tmc_code'].nunique():,} TMCs")
else:
    print("✗ No data after filtering - threshold may be too high or data quality issues")





Using minimum epoch threshold: 25
After filtering: 55,112 rows, 2,833 TMCs
✓ Hourly statistics calculated successfully!
Hourly statistics calculated for 2,833 TMCs


*** 
Calculate Worst 4 Hours Speed (Temp Table #most_congd_hrs)
***

In [15]:
# Get worst 4 hours for each TMC
worst_hours = hourly_stats[hourly_stats['hour_cong_rank'] < 5][['tmc_code', 'hour']]

# Filter traffic data for worst hours only
df_worst = df_tmc.merge(df_traffic, left_on='tmc', right_on='tmc_code', how='inner')
df_worst['day_name'] = df_worst['measurement_tstamp'].dt.day_name()
df_worst = df_worst[df_worst['day_name'].isin(WEEKDAYS)]
df_worst['hour'] = df_worst['measurement_tstamp'].dt.hour

# Join to get only worst hours
df_worst = df_worst.merge(worst_hours, on=['tmc_code', 'hour'], how='inner')
df_worst = df_worst.merge(ff_speeds, on='tmc_code', how='inner')

# Calculate harmonic average for worst 4 hours
worst_stats = df_worst.groupby('tmc_code').agg(
    epochs_worst4hrs=('measurement_tstamp', 'count'),
    havg_spd_worst4hrs=('speed', lambda x: len(x) / (1.0 / x).sum()),
    ff_speed_art60thp=('ff_speed_art60thp', 'first')
).reset_index()

print(f"Worst 4 hours calculated for {len(worst_stats):,} TMCs")

Worst 4 hours calculated for 2,833 TMCs


*** 
Find Slowest Hour (Temp Table #slowest_hr)
***

In [16]:
slowest = hourly_stats[hourly_stats['hour_cong_rank'] == 1].copy()
slowest = slowest[['tmc_code', 'hour', 'havg_spd_weekdy', 'total_epochs_hr']]
slowest.columns = ['tmc_code', 'slowest_hr', 'slowest_hr_speed', 'epochs_slowest_hr']

# Remove duplicates (keep first if multiple hours tied)
slowest = slowest.drop_duplicates(subset=['tmc_code'], keep='first')

print(f"Slowest hour identified for {len(slowest):,} TMCs")


Slowest hour identified for 2,833 TMCs


*** 
Create Final Report (Temp Table #data_tmc_final)
***

In [17]:
# Rename tmc_code to tmc in all source dataframes first
ff_speeds = ff_speeds.rename(columns={'tmc_code': 'tmc'})
worst_stats = worst_stats.rename(columns={'tmc_code': 'tmc'})
slowest = slowest.rename(columns={'tmc_code': 'tmc'})
epochs_night = epochs_night.rename(columns={'tmc_code': 'tmc'})

# Then merge cleanly on 'tmc'
final = df_tmc[df_tmc['nhs'] > 0].copy()
final = final.merge(ff_speeds, on='tmc', how='left')
final = final.merge(worst_stats[['tmc', 'havg_spd_worst4hrs', 'epochs_worst4hrs']], 
                   on='tmc', how='left')
final = final.merge(slowest, on='tmc', how='left')
final = final.merge(epochs_night, on='tmc', how='left')

# Fill missing values with -1
metric_cols = ['ff_speed_art60thp', 'havg_spd_worst4hrs', 'slowest_hr', 
               'slowest_hr_speed', 'epochs_worst4hrs', 'epochs_slowest_hr', 'epochs_night']
for col in metric_cols:
    if col in final.columns:
        final[col] = final[col].fillna(-1.0)

# Calculate congestion ratios
final['congratio_worst4hrs'] = np.where(
    (final['havg_spd_worst4hrs'] > -1) & (final['ff_speed_art60thp'] > -1),
    np.minimum(final['havg_spd_worst4hrs'] / final['ff_speed_art60thp'], 1.0),
    -1.0
)

final['congratio_worsthr'] = np.where(
    (final['slowest_hr_speed'] > -1) & (final['ff_speed_art60thp'] > -1),
    final['slowest_hr_speed'] / final['ff_speed_art60thp'],
    -1.0
)
final['year_month'] = year_month
print(f"Final report created for {len(final):,} NHS TMCs")

Final report created for 2,905 NHS TMCs


*** 
Calculate System wide metrics
***

In [18]:
# Filter valid data
valid_mask = (final['havg_spd_worst4hrs'] > -1) & (final['ff_speed_art60thp'] > -1)
tot_nhs_dirmiles = final[valid_mask]['miles'].sum()

# Miles where congestion < 60% of free-flow
congested_mask = (final['havg_spd_worst4hrs'] / final['ff_speed_art60thp'] < 0.6) & valid_mask
final[congested_mask].head()
congested_miles = final[congested_mask]['miles'].sum()

pct_dirmi_congested = (congested_miles / tot_nhs_dirmiles * 100) if tot_nhs_dirmiles > 0 else 0

print(f"\nTotal NHS directional miles: {tot_nhs_dirmiles:,.2f}")
print(f"Congested miles (<60% free-flow): {congested_miles:,.2f}")
print(f"Percent of miles congested: {pct_dirmi_congested:.2f}%")

# Show TMCs with missing data
missing_data = final[(final['havg_spd_worst4hrs'] == -1) | (final['ff_speed_art60thp'] == -1)]
print(f"\nTMCs with insufficient data: {len(missing_data):,}")

# Count tmcs with valid speed metrics for congestion calculation
num_valid_tmcs = valid_mask.sum()
print(f"TMCs with valid speed metrics: {num_valid_tmcs:,}")
# Total observations used in the congestion calculation (only count observations from TMCs that were actually used and have epoch data)
obs_mask = valid_mask & (final['epochs_worst4hrs'] > -1)
obs_used_for_congestion = int(final[obs_mask]['epochs_worst4hrs'].sum())

print(f"Total observations used for congestion calculation: {obs_used_for_congestion:,}")

 # Store results for final summary
system_metrics = {
    'month': year_month,
    'total_nhs_dirmiles': tot_nhs_dirmiles,
    'congested_miles': congested_miles,
    'pct_miles_congested': pct_dirmi_congested,
    'tmcs_insufficient_data': len(final[~valid_mask]),
    'num_valid_tmcs': num_valid_tmcs,
    'obs_used_for_congestion': obs_used_for_congestion
}
df_summary = pd.DataFrame([system_metrics])
# Reorder columns for a cleaner look
# Remove this column selection, or add the missing columns:
df_summary = df_summary[['month', 'total_nhs_dirmiles', 'congested_miles', 'pct_miles_congested', 
                         'tmcs_insufficient_data', 'num_valid_tmcs', 'obs_used_for_congestion']]


Total NHS directional miles: 1,990.43
Congested miles (<60% free-flow): 468.19
Percent of miles congested: 23.52%

TMCs with insufficient data: 72
TMCs with valid speed metrics: 2,833
Total observations used for congestion calculation: 1,040,336


In [19]:

#valid_mask.sum()

*** 
Sample Results
*** 

In [20]:
print("\nSample of Final Report (first 10 rows):")
display_cols = ['tmc', 'road', 'route_numb', 'f_system', 'miles', 
                'ff_speed_art60thp', 'havg_spd_worst4hrs', 'congratio_worst4hrs',
                'slowest_hr', 'slowest_hr_speed']
print(final[display_cols].head(10).to_string(index=False))



Sample of Final Report (first 10 rows):
      tmc           road  route_numb  f_system    miles  ff_speed_art60thp  havg_spd_worst4hrs  congratio_worst4hrs  slowest_hr  slowest_hr_speed
105-04687          US-50        50.0       2.0 0.169567             59.850           28.396041             0.474453        17.0         24.512381
105-04688          US-50        50.0       2.0 0.933931             63.992           23.355509             0.364975        16.0         21.013661
105-04686          US-50        50.0       2.0 0.017779             62.640           28.163300             0.449606        17.0         23.006158
105-04689          US-50        50.0       2.0 0.200784             65.378           23.576834             0.360623        16.0         20.152462
105-16670 ELDER CREEK RD         NaN       3.0 0.445003             27.118           14.982909             0.552508        16.0         11.638850
105-16672 ELDER CREEK RD         NaN       3.0 1.001903             29.000         

*** 
Export
*** 

In [21]:
# Define the output file name for the final report
base_filename = zip_name.split(".")[0]
output_filename = f"{base_filename}_final{tp_dict[truck_pax_combined]}.csv"
output_path = PATH_FINAL / output_filename
    
# Check if the output file already exists
if output_path.exists():
    print(f"\n*** Skipping {zip_name} ***")
    print(f"  Output file already exists at: {output_path}")
else:
    print(f"  ...Exporting to {output_filename}")
    final.to_csv(output_path, index=False)
    print("  Export complete.")


# Define the output file name for the monthly summary
summary_output_filename = f"{base_filename}_summary{tp_dict[truck_pax_combined]}.csv"
summary_output_path = PATH_SUMMARY / summary_output_filename
    
# Check if the output file already exists
if summary_output_path.exists():
    print(f"\n*** Skipping {zip_name} ***")
else:
    print(f"  ...Exporting to {summary_output_filename}")
    df_summary.to_csv(summary_output_path, index=False)
    print("  Export complete.")
    


*** Skipping 2025_01TAP.zip ***
  Output file already exists at: I:\Projects\Josh\Regional Monitoring\Congestion\final csv\2025_01TAP_finalTAP.csv

*** Skipping 2025_01TAP.zip ***
